# Problem 3 — Phase 4: Kernel pilot (not full tree)

**Index:** [`hierarchical_problem3_index.ipynb`](hierarchical_problem3_index.ipynb) · **Prev:** [Phase 3](hierarchical_problem3_phase3_by_depth.ipynb)

**Main pilot:** sample **20 random binary edges** (same eligibility as Phase 3: train has two classes, subtree-restricted pool). For each edge, subsample **train** for fitting, then score **F1 on the test** split (**capped** per edge via `MAX_TEST_SUBSAMPLE` so RBF `SVC` stays fast); report **macro** (mean over edges) and **pooled micro** F1. With **`VERBOSE_TRAIN = True`**, logs match Phase 2/3: **`PHASE 4 MODEL [i/3]`**, per-edge **`fit: …`**, **`>>> … finished`**.

[`pilot_kernel_compare_on_edges`](hierarchical_summary_metrics.py) compares:

- **Linear** — `TfidfVectorizer` + `LinearSVC` (same spirit as the full-tree linear router).
- **RBF** — TF-IDF + `SVC` (RBF kernel).
- **Explicit degree-2** — TF-IDF → `TruncatedSVD` → `PolynomialFeatures(degree=2)` → `LinearSVC` (squares and pairwise interactions of SVD components; cheap explicit nonlinear features).

Optional single-edge helper: [`pilot_kernel_svc_on_edge`](hierarchical_summary_metrics.py) (internal train/val split, `SVC` linear / RBF / built-in poly kernel on one edge).


In [2]:
from __future__ import annotations

import json
import random
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

from hierarchical_summary_metrics import pilot_kernel_compare_on_edges, pilot_kernel_svc_on_edge
from hierarchical_training_data import make_multilabel_binary_pool_data
from topic_hierarchy import BinaryEdgeSpec, TopicTree, binary_edge_specs, load_topic_tree, summary


def homework4_base() -> Path:
    here = Path.cwd().resolve()
    if (here / "topics.csv").exists():
        return here
    for p in [here, *here.parents]:
        if (p / "topics.csv").exists():
            return p
    raise FileNotFoundError("topics.csv not found — cwd should be Homework 4")


BASE = homework4_base()
tree = load_topic_tree(str(BASE / "topics.csv"))
mldata = make_multilabel_binary_pool_data(base_path=str(BASE))
RESTRICT = True
MAX_FEATURES = 8000

N_PILOT_EDGES = 20
PILOT_SEED = 42
MAX_TRAIN_SUBSAMPLE = 3000
MAX_TEST_SUBSAMPLE = 1500  # cap test docs per edge (stratified); set None for full test
SVD_COMPONENTS = 100
VERBOSE_TRAIN = True  # PHASE 4 MODEL [...] + per-edge lines (same spirit as Phase 2/3)

print("BASE", BASE)
print(summary(tree))
print("train", len(mldata.train_ids()), "test", len(mldata.test_ids()))


BASE /Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4
{'traversal_root': 'Root', 'n_nodes': 117, 'n_leaves': 82, 'n_branching_nodes': 22, 'n_local_classifiers': 22, 'n_binary_edge_classifiers': 103, 'max_branching_factor': 23, 'max_depth_from_root': 5}
train 14465 test 3617


In [3]:
def eligible_specs_for_pilot(pool, tree: TopicTree, *, restrict: bool) -> list[BinaryEdgeSpec]:
    out: list[BinaryEdgeSpec] = []
    for spec in binary_edge_specs(tree):
        _, ytr = pool.binary_edge_dataset(
            spec.parent,
            spec.child,
            "train",
            restrict_to_parent_subtree=restrict,
        )
        if len(set(ytr)) >= 2:
            out.append(spec)
    return out


eligible = eligible_specs_for_pilot(mldata, tree, restrict=RESTRICT)
rng = random.Random(PILOT_SEED)
k = min(N_PILOT_EDGES, len(eligible))
pilot_specs = rng.sample(eligible, k=k) if k else []
pilot_tuples = [(s.parent, s.child) for s in pilot_specs]

pilot_path = BASE / "problem3_phase4_pilot_edges.json"
with open(pilot_path, "w") as f:
    json.dump(
        [{"parent": s.parent, "child": s.child, "depth": s.depth} for s in pilot_specs],
        f,
        indent=2,
    )
print(f"Eligible {len(eligible)}; pilot {len(pilot_tuples)} edges -> {pilot_path}")

if not pilot_tuples:
    print("No pilot edges.")
else:
    print("\n" + "=" * 72, flush=True)
    print(
        f"PHASE 4: kernel compare on {len(pilot_tuples)} pilot edges  "
        f"(max_train_fit={MAX_TRAIN_SUBSAMPLE}, max_test={MAX_TEST_SUBSAMPLE} per edge)",
        flush=True,
    )
    print("=" * 72 + "\n", flush=True)
    t0 = time.perf_counter()
    summary_df, detail = pilot_kernel_compare_on_edges(
        mldata,
        pilot_tuples,
        max_train=MAX_TRAIN_SUBSAMPLE,
        max_test=MAX_TEST_SUBSAMPLE,
        max_features=MAX_FEATURES,
        svd_components=SVD_COMPONENTS,
        random_state=PILOT_SEED,
        restrict_to_parent_subtree=RESTRICT,
        verbose=VERBOSE_TRAIN,
    )
    print(
        f"\n>>> Phase 4 multi-edge pilot done in {time.perf_counter() - t0:.1f}s wall time\n",
        flush=True,
    )
    display(summary_df)
    baseline = summary_df.loc["linear_LinearSVC", "pooled_micro_f1_test"]
    for name in summary_df.index:
        p = summary_df.loc[name, "pooled_micro_f1_test"]
        delta = p - baseline
        print(f"  {name}: pooled F1={p:.4f}  (Δ vs linear {delta:+.4f})")


Eligible 101; pilot 20 edges -> /Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4/problem3_phase4_pilot_edges.json

PHASE 4: kernel compare on 20 pilot edges  (max_train_fit=3000, max_test=1500 per edge)


PHASE 4 MODEL [1/3] linear_LinearSVC  (20 pilot edges)
  [1/20] E21 → E212  fit: n_train_fit=765  n_test=171 ...
    ok  f1_test=0.8966  (0.3s)
  [2/20] ECAT → E7  fit: n_train_fit=3000  n_test=1088 ...
    ok  f1_test=0.8684  (0.9s)
  [3/20] Root → MCAT  fit: n_train_fit=3000  n_test=1500 ...
    ok  f1_test=0.7797  (1.2s)
  [4/20] G15 → G157  fit: n_train_fit=1499  n_test=362 ...
    ok  f1_test=0.7805  (1.5s)
  [5/20] GCAT → GWEA  fit: n_train_fit=3000  n_test=1500 ...
    ok  f1_test=0.8276  (2.2s)
  [6/20] GCAT → GSPO  fit: n_train_fit=3000  n_test=1500 ...
    ok  f1_test=0.7937  (1.2s)
  [7/20] GCAT → GPRO  fit: n_train_fit=3000  n_test=1500 ...
    ok  f1_test=0.6222  (2.4s)
  [8/20] GCAT → GDEF  fit: n_train_fit=3000  n_test=15

,macro_f1_test,pooled_micro_f1_test,n_edges_scored,total_fit_eval_sec
model,,,,
linear_LinearSVC,0.818500,0.823691,20,25.346291
RBF_SVC,0.747709,0.797970,20,150.515262
explicit_poly2,0.795597,0.822785,20,263.971228


  linear_LinearSVC: pooled F1=0.8237  (Δ vs linear +0.0000)
  RBF_SVC: pooled F1=0.7980  (Δ vs linear -0.0257)
  explicit_poly2: pooled F1=0.8228  (Δ vs linear -0.0009)


### Notes

- **Explicit poly** uses a degree-2 polynomial in an **SVD** subspace of TF-IDF (not full-vocabulary \(x_i x_j\) products, which would be enormous). It is a lightweight stand-in for “quadratics + interactions.”
- If linear ≈ nonlinear F1 but RBF / poly are much slower per fit, prefer **linear `LinearSVC`** for full-tree training. Full-tree kernel SVMs are usually impractical here.

### Optional: single-edge `SVC` kernel sweep (train/val)

Same helper as before: one Root→H1 edge with subsampled train and an internal validation split; compares sklearn **`SVC`** linear / RBF / built-in polynomial **kernel** (not the explicit SVD+Poly pipeline above).


In [ ]:
root = tree.traversal_root
children = tree.children.get(root, [])
pilot_edge = None
for ch in children:
    Xtr, ytr = mldata.binary_edge_dataset(root, ch, "train")
    if len(set(ytr)) >= 2 and len(Xtr) >= 100:
        pilot_edge = (root, ch)
        break
if pilot_edge:
    pilot_res = pilot_kernel_svc_on_edge(
        mldata, pilot_edge[0], pilot_edge[1], verbose=VERBOSE_TRAIN
    )
    display(pd.DataFrame(pilot_res).T)
else:
    print("No suitable H1 edge for single-edge pilot")